In [1]:
from dataclasses import dataclass
import torch
import torch.nn as nn
import json
from tqdm import tqdm
from core.config import Config, InferenceConfig

In [2]:
class Inference:
    def __init__(self,
                 model: nn.Module,
                 tokenizer,
                 config: InferenceConfig,
    ):
        self.model = model.to(config.device)
        self.tokenizer = tokenizer
        self.config = config
        self.model.eval()
        print(f"Using : {config.device}")
    def load(self, checkpoint_path: str):
        checkpoint = torch.load(
            checkpoint_path,
            map_location=self.config.device,
        )

        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint

        self.model.load_state_dict(state_dict)
        self.model.eval()

    def __sample_next_token(self, logits):
        if self.config.sampling_method == "argmax":
            return torch.argmax(logits,dim=-1).reshape(1)
        else:
            # Temperature
            logits = logits / self.config.temperature
            # Top-k sampling
            if self.config.top_k is not None:
                values, indices = torch.topk(
                    logits,
                    self.config.top_k,
                    dim=-1
                )
                # Mask For -inf
                mask = torch.ones_like(logits,dtype=torch.bool)
                mask.scatter_(dim=-1, index=indices, value=False)
                logits[mask] = -torch.inf
            # Convert to probability
            probs = torch.softmax(logits, dim=-1)

            if self.config.top_p is not None:
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
                sorted_indices_to_remove = cumulative_probs > self.config.top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = False
                sorted_probs[sorted_indices_to_remove] = 0.0
                sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
                # Sample from sorted probabilities
                sampled = torch.multinomial(sorted_probs, num_samples=1)
                # Convert sorted position -> original token ID
                next_token = torch.gather(
                    sorted_indices,
                    -1,
                    sampled
                )
            else:
                # Sample next token
                next_token = torch.multinomial(probs, num_samples=1)

            return next_token

    @torch.inference_mode
    def generate(self, prompt: str) -> str:
        # Encode prompt
        input_ids = torch.tensor(self.tokenizer.encode(
            prompt,
        )).unsqueeze(0).to(self.config.device)
        generated_ids = input_ids.clone()

        # No cache initially
        kv_cache = None

        for _ in range(self.config.max_tokens-len(input_ids)):
            if kv_cache is not None:
                model_input = generated_ids[:, -1:]
            else:
                model_input = generated_ids

            # Forward pass
            logits,kv_cache = self.model(model_input,kv_cache=kv_cache)
            # Last token logits
            logits = logits[0, -1, :]
            next_token = self.__sample_next_token(logits)
            next_token = next_token.unsqueeze(0) # Match the Batch Dimension
            # Store generated token
            generated_ids = torch.cat(
                [generated_ids, next_token],
                dim=-1
            )
            # EOS
            if (
                self.config.eos_token_id is not None
                and next_token.item() == self.config.eos_token_id
            ):
                break
        return self.tokenizer.decode(
            generated_ids.squeeze().tolist()
        )
    def generate_multiple(self,prompts:list[str])->list[str]:
        l = []
        pbar = tqdm(prompts,desc="Generating Responses")
        for i,p in enumerate(pbar):
            l.append(self.generate(p))
            pbar.set_postfix({"count":i+1})
        return l

In [3]:
from peak.core.tokenizer import BPE
from peak.core.peakmodel import PeakModel
from core.config import Config
configs = Config.from_json("exp_1/exp_1_config.json")

model = PeakModel(configs.model,is_inference_mode=True)
tokenizer = BPE.load('exp_1/tokenizer','peak_base_1')

In [4]:
c = 0
for i in model.parameters():
    c+=i.numel()
print(f"Model Has {c // 1_000_000}M Parameters")

Model Has 12M Parameters


In [5]:
infconfig = InferenceConfig(
    max_tokens=500,
    temperature=1.0,
    top_k=None,
    top_p=1.0,
    sampling_method="argmax",
    eos_token_id=256,
    device="cuda"
)

inference = Inference(
    model=model,
    tokenizer=tokenizer,
    config=infconfig,
)
try :
    inference.load("exp_1/checkpoints/best.pt")
except RuntimeError:
    print("Weights not loaded Successfully....")

Using : cuda


In [6]:
tokens = tokenizer.encode("The cat climbed up the tree because it was scared of the dog.")
print("TOKEN COUNT:", len(tokens))
print("TOKENS:", tokens)

TOKEN COUNT: 14
TOKENS: [377, 708, 1577, 362, 265, 685, 795, 307, 284, 637, 351, 265, 640, 46]


In [8]:
text = inference.generate(
    "The cat climbed up the tree because it was scared of the dog.",
)
print(text)

The cat climbed up the tree because it was scared of the dog................................


In [29]:
# with open("testing/test_prompts.json") as f:
#     prompts = json.load(f)["prompts"]

# save_model_outputs(prompts,inference.generate_multiple(prompts),"testing/outputs.json",extra_info={
#         "parameters": "7M",
#         "context_length": 512,
#         "temperature": config.temperature,
#         "top_k": config.top_k,
#         "top_p": config.top_p,
# })

Generating Responses: 100%|██████████| 49/49 [01:00<00:00,  1.24s/it, count=49]

Saved 49 samples to: metrics\outputs.json
